# Module 2: Trident Installation

## Exercise 1: Installing Trident

In this exercise, you install NetApp Trident by using the manual operator method. 

You can also use Helm or the tridentctl method to install the Trident operator, but this exercise does not discuss
these other approaches.

**Objectives**

This exercise focuses on enabling you to do the following:

  - Download and set up the Trident operator
  - Deploy instances of Trident
  - Set up the tridentctl tool
  - Prepare worker nodes

**Exercise Equipment**

In this exercise, you use the following systems.

| System                  | Host Name   | IP Addresses   | User Name (case sensitive) | Password  |
|-------------------------|-------------|----------------|----------------------------|-----------|
| Linux Mint 20           | jumphost    | 192.168.0.5    | user                       | Netapp1!  |
| Kubernetes Control Plane| kubmas1-1   | 192.168.0.61   | root                       | Netapp1!  |
| Kubernetes Worker 1     | kubwor1-1   | 192.168.0.62   | root                       | Netapp1!  |
| Kubernetes Worker 2     | kubwor1-2   | 192.168.0.63   | root                       | Netapp1!  |
| Kubernetes Worker 3     | kubwor1-3   | 192.168.0.64   | root                       | Netapp1!  |

**Prerequisites**

Before starting this exercise, you should take the following actions:

  -  Set up your Integrated Development Environment (IDE)
  -  Download the courseware GIT repository
  -  Configure your IDE to have access to your Kubernetes clusters
  -  Create svm0
  -  Configure svm0 to use the NFS v3 protocol


---
---


#### Task 1: Download and set Up the Trident operator

In this task, you verify that you can access the Kubernetes cluster, 

and you download and set up the Trident operator.


---

If desired, you can follow along with this exercise on the Trident operator deployment:

https://docs.netapp.com/us-en/trident/trident-get-started/kubernetes-deploy-operator.html#deploy-the-trident-operator-manually 

---

Verify that you have administrative access to the Kubernetes cluster:


In [2]:
kubectl auth can-i '*' '*' --all-namespaces


yes


---

In a future exercise, you implement Container Storage Interface (CSI) topologies. 

To support this effort, you apply different labels to each worker node. 

These labels should be present on the nodes in the cluster before you install Trident. 

The labels enable Trident to be topology-aware.


---

Label Worker 1 as Zone 1 and a region (for convenience, see [exercise2Task1-1.txt](./exercise2Task1-1.txt)):



In [3]:
kubectl label node kubwor1-1 topology.kubernetes.io/region=trident topology.kubernetes.io/zone=zone1


node/kubwor1-1 labeled


---

Label Worker 2 as Zone 2 and a region (for convenience, see [exercise2Task1-2.txt](./exercise2Task1-2.txt)):



In [4]:
kubectl label node kubwor1-2 topology.kubernetes.io/region=trident topology.kubernetes.io/zone=zone2

node/kubwor1-2 labeled


---

Label Worker 3 as Zone 3 and a region (for convenience, see [exercise2Task1-3.txt](./exercise2Task1-3.txt)):


In [5]:
kubectl label node kubwor1-3 topology.kubernetes.io/region=trident topology.kubernetes.io/zone=zone3

node/kubwor1-3 labeled


---

Use a web browser to navigate to https://github.com/Netapp/trident/releases.

---

Identify the latest version of Trident at the top of the page

---

If desired, you can download a newer version. 

However, this exercise is not tested with any version other than 24.10.0. 

If you want to work with this version of the exercise, you can find the tar.gz file in the Exercise 2 folder in your class files.


---

Verify that you are in the `./Exercise 2`

In [6]:
pwd

/home/user/STRSW-ILT-UATWK-1/Exercise 2


Unzip the Trident file:



In [7]:
tar -xf trident-installer-24.10.0.tar.gz


A new subfolder, called trident-installer, should appear under the Exercise 2 folder.


---

cd to  Exercise 2 > trident-installer folder.

NOTE: This path serves as the relative path for all other paths in this task and the next task.


In [8]:
cd trident-installer
pwd

/home/user/STRSW-ILT-UATWK-1/Exercise 2/trident-installer


---

Investigate the deploy/crds subfolder.


In [9]:
ls -l deploy/crds

total 36
-rw-r--r-- 1 user user 1485 Oct 31  2024 trident.netapp.io_tridentconfigurators_crd.yaml
-rw-r--r-- 1 user user  585 Oct 31  2024 trident.netapp.io_tridentorchestrators_crd_post1.16.yaml
-rw-r--r-- 1 user user  585 Oct 31  2024 trident.netapp.io_tridentorchestrators_crd.yaml
-rw-r--r-- 1 user user  168 Oct 31  2024 tridentorchestrator_cr_audit_log.yaml
-rw-r--r-- 1 user user  261 Oct 31  2024 tridentorchestrator_cr_autosupport.yaml
-rw-r--r-- 1 user user  178 Oct 31  2024 tridentorchestrator_cr_customimage.yaml
-rw-r--r-- 1 user user  179 Oct 31  2024 tridentorchestrator_cr_default.yaml
-rw-r--r-- 1 user user  203 Oct 31  2024 tridentorchestrator_cr_imagepullsecrets.yaml
-rw-r--r-- 1 user user  195 Oct 31  2024 tridentorchestrator_cr.yaml


---

The crds subfolder contains several custom resource definition (CRD) YAML files.

Notice that three custom resource definitions (CRD) files with crd in the filenames

and six of these custom resources files with cr in the filenames.


---

Create the CRD definitions by using the `trident.netapp.io_tridentorchestrators_crd_post1.16.yaml` file:


In [10]:
kubectl create -f deploy/crds/trident.netapp.io_tridentorchestrators_crd_post1.16.yaml


customresourcedefinition.apiextensions.k8s.io/tridentorchestrators.trident.netapp.io created


---

In the Kubernetes Extension of your IDE, you should see the
`tridentorchestrators` CRD.

Expand **Clusters**> **source-admin@source**> **Custom Resources** to view in the Kubernetes Extension.

If you see an error under the tridentorchestrators CRD, click the Refresh button to
make it disappear.


---

Create the trident namespace:


In [11]:
kubectl create -f deploy/namespace.yaml


namespace/trident created


---

Copy and rename the resulting file for the aggregated YAML “kustomized” file for Kubernetes
version 1.25 or later (for convenience, see [exercise2Task1-4.txt](./exercise2Task1-4.txt)):


In [12]:

cp deploy/kustomization_post_1_25.yaml deploy/kustomization.yaml


NOTE: 

This kustomization.yaml file runs the 

- serviceaccount.yaml, 
- clusterrolebinding.yaml, and the 
- operator.yaml 

files.


---

Create a YAML bundle that you can run (for convenience, see [exercise2Task1-5.txt](./exercise2Task1-5.txt)):



In [13]:
kubectl kustomize deploy/ > deploy/bundle_post_1_25.yaml

---

Install the operator (for convenience, see [exercise2Task1-6.txt](./exercise2Task1-6.txt)):



In [14]:
kubectl create -f deploy/bundle_post_1_25.yaml

serviceaccount/trident-operator created
clusterrole.rbac.authorization.k8s.io/trident-operator created
clusterrolebinding.rbac.authorization.k8s.io/trident-operator created
deployment.apps/trident-operator created


---

Verify that you created all the objects:

`kubectl -n trident get all`

Sample output:

```terminal
NAME READY STATUS RESTARTS AGE
pod/trident-operator-5c94fc5556-nlsnl 1/1 Running 0 2m7s
NAME READY UP-TO-DATE AVAILABLE AGE
deployment.apps/trident-operator 1/1 1 1 2m7s
NAME DESIRED CURRENT READY AGE
replicaset.apps/trident-operator-5c94fc5556 1 1 1 2m7s
```

In [15]:
kubectl -n trident get all

NAME                                    READY   STATUS              RESTARTS   AGE
pod/trident-operator-74978c48ff-t6qd2   0/1     ContainerCreating   0          19s

NAME                               READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/trident-operator   0/1     1            0           19s

NAME                                          DESIRED   CURRENT   READY   AGE
replicaset.apps/trident-operator-74978c48ff   1         1         0       19s


In [16]:
kubectl -n trident get all

NAME                                    READY   STATUS    RESTARTS   AGE
pod/trident-operator-74978c48ff-t6qd2   1/1     Running   0          70s

NAME                               READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/trident-operator   1/1     1            1           70s

NAME                                          DESIRED   CURRENT   READY   AGE
replicaset.apps/trident-operator-74978c48ff   1         1         1       70s


---

A Kubernetes cluster should contain only one instance of the operator. 

You must not create multiple deployments of the Trident operator.


---
---

#### Task 2: Deploy instances of Trident

In this task, you use the operator to deploy Trident. 

This action requires you to create a `TridentOrchestrator` custom resource (CR). 

The Trident installer includes example definitions for creating the `TridentOrchestrator` CR. 

This CR starts an installation in the trident namespace.

The relative path for this exercise is the Exercise 2 > trident-installer folder.



---

Review the [deploy/crds/tridentorchestrator_cr.yaml](./trident-installer/deploy/crds/tridentorchestrator_cr.yaml) file.


---

Create an instance of the `TridentOrchestrator` CR:


In [17]:
kubectl create -f deploy/crds/tridentorchestrator_cr.yaml


tridentorchestrator.trident.netapp.io/trident created


---

In the Kubernetes Extension of your IDE, you should see the trident instance
under the `tridentorchestrators` CRD.


---

Review the details by double-clicking the trident entry in the hierarchy, or use the following
command:


In [18]:

kubectl -n trident describe torc trident


Name:         trident
Namespace:    
Labels:       <none>
Annotations:  <none>
API Version:  trident.netapp.io/v1
Kind:         TridentOrchestrator
Metadata:
  Creation Timestamp:  2025-07-08T18:54:01Z
  Generation:          1
  Resource Version:    226234
  UID:                 2d4e3aa8-b2a0-4d0b-ae71-84df3c0ef3f9
Spec:
  Cloud Provider:     
  Debug:              true
  Image Pull Policy:  IfNotPresent
  Namespace:          trident
  Windows:            false
Status:
  Current Installation Params:
    IPv6:                          
    Acp Image:                     
    Autosupport Hostname:          
    Autosupport Image:             
    Autosupport Insecure:          false
    Autosupport Proxy:             
    Autosupport Serial Number:     
    Debug:                         
    Disable Audit Log:             
    Enable ACP:                    
    Enable Force Detach:           
    Http Request Timeout:          
    Image Pull Policy:             
    Image Pull Secrets

In [19]:

kubectl -n trident describe torc trident


Name:         trident
Namespace:    
Labels:       <none>
Annotations:  <none>
API Version:  trident.netapp.io/v1
Kind:         TridentOrchestrator
Metadata:
  Creation Timestamp:  2025-07-08T18:54:01Z
  Generation:          1
  Resource Version:    226758
  UID:                 2d4e3aa8-b2a0-4d0b-ae71-84df3c0ef3f9
Spec:
  Cloud Provider:     
  Debug:              true
  Image Pull Policy:  IfNotPresent
  Namespace:          trident
  Windows:            false
Status:
  Acp Version:  v24.10.0
  Current Installation Params:
    IPv6:                       false
    Acp Image:                  
    Autosupport Hostname:       
    Autosupport Image:          docker.io/netapp/trident-autosupport:24.10
    Autosupport Insecure:       false
    Autosupport Proxy:          
    Autosupport Serial Number:  
    Debug:                      true
    Disable Audit Log:          true
    Enable ACP:                 false
    Enable Force Detach:        false
    Http Request Timeout:       90s
 

---

Answer the following question:

In the events section, what is the last event type and reason?


---

Verify that you created all objects:


Sample output:

```terminal
NAME READY STATUS RESTARTS AGE
pod/trident-controller-74698976f5-5d2tz 6/6 Running 0 3m16s
pod/trident-node-linux-8t2tp 2/2 Running 2 (60s ago) 3m16s
pod/trident-node-linux-dnbm6 2/2 Running 2 (77s ago) 3m16s
pod/trident-node-linux-k29t7 2/2 Running 2 (76s ago) 3m16s
pod/trident-node-linux-td97m 2/2 Running 2 (71s ago) 3m16s
pod/trident-operator-5c94fc5556-nlsnl 1/1 Running 0 13h
NAME TYPE CLUSTER-IP EXTERNAL-IP PORT(S)
service/trident-csi ClusterIP 10.102.83.55 <none> 34571/TCP,9220/TCP 3
NAME DESIRED CURRENT READY UP-TO-DATE AVAIL
daemonset.apps/trident-node-linux 4 4 4 4 4
NAME READY UP-TO-DATE AVAILABLE AGE
deployment.apps/trident-controller 1/1 1 1 3m16s
deployment.apps/trident-operator 1/1 1 1 13h
NAME DESIRED CURRENT READY AGE
replicaset.apps/trident-controller-74698976f5 1 1 1 3m16s
replicaset.apps/trident-operator-5c94fc5556 1 1 1 13h
```

In [20]:
kubectl -n trident get all


NAME                                      READY   STATUS    RESTARTS       AGE
pod/trident-controller-66b97d5f8d-z68kc   6/6     Running   0              3m26s
pod/trident-node-linux-7l8cb              2/2     Running   2 (111s ago)   3m26s
pod/trident-node-linux-8kzhz              2/2     Running   3 (81s ago)    3m26s
pod/trident-node-linux-lpqnf              2/2     Running   3 (78s ago)    3m26s
pod/trident-node-linux-ls577              2/2     Running   3 (78s ago)    3m26s
pod/trident-operator-74978c48ff-t6qd2     1/1     Running   0              9m28s

NAME                  TYPE        CLUSTER-IP       EXTERNAL-IP   PORT(S)              AGE
service/trident-csi   ClusterIP   10.102.243.220   <none>        34571/TCP,9220/TCP   3m31s

NAME                                DESIRED   CURRENT   READY   UP-TO-DATE   AVAILABLE   NODE SELECTOR   AGE
daemonset.apps/trident-node-linux   4         4         4       4            4           <none>          3m26s

NAME                          

---

The DaemonSet `trident-node-linux` creates the four `trident-node-linux` pods. 

One `trident-node-linux` pod is installed on each node (including the control-plane master node). 

The `trident-controller` deployment creates the `trident-controller` pod, which runs on one of the worker nodes.


---

Stop the deployed Trident pods by deleting the TridentOrchestrator CR:


In [21]:
kubectl -n trident delete torc trident


tridentorchestrator.trident.netapp.io "trident" deleted


---

Verify that every pod with node or controller in its name is deleted and that only the
Trident operator is running:


In [22]:
kubectl -n trident get pods -o wide


NAME                                READY   STATUS    RESTARTS   AGE   IP          NODE        NOMINATED NODE   READINESS GATES
trident-operator-74978c48ff-t6qd2   1/1     Running   0          12m   10.42.0.1   kubwor1-3   <none>           <none>


The TridentOrchestrator CR enables you to customize the Trident operator.

See the following URL for more details: 

https://docs.netapp.com/us-en/trident/trident-get-started/kubernetes-customize-deploy.html.

The “crds” subfolder contains several examples of modifications.


---

Try to run Trident only on Worker 1 and Worker 3.

---

Create labels on two worker nodes:


In [23]:
kubectl label node kubwor1-1 storage=trident
kubectl label node kubwor1-3 storage=trident


node/kubwor1-1 labeled
node/kubwor1-3 labeled


---

Edit the [deploy/crds/tridentorchestrator_cr.yaml](./trident-installer/deploy/crds/tridentorchestrator_cr.yaml) file to add an appropriate toleration:

(see https://docs.netapp.com/us-en/trident/trident-get-started/kubernetes-customize-deploy.html)

 -  Definition: **nodePluginNodeSelector** 
 -  Key: storage
 -  Value: trident

<details> <summary>Solution  </summary>

You can find the solution for this step in the [exercise2Task2-nodeselector.yaml](./Solutions/exercise2Task2-nodeselector.yaml) file.


---

Create an instance of the TridentOrchestrator CR:


In [24]:
kubectl create -f ../Solutions/exercise2Task2-nodeselector.yaml

tridentorchestrator.trident.netapp.io/trident created


---

After a few minutes, verify that the trident controller and node pods are only running on
kubwor1-1 and kubwor1-3:


In [25]:
kubectl -n trident get pods -o wide


NAME                                  READY   STATUS    RESTARTS   AGE   IP             NODE        NOMINATED NODE   READINESS GATES
trident-controller-66b97d5f8d-mjnwg   6/6     Running   0          46s   10.36.0.1      kubwor1-1   <none>           <none>
trident-node-linux-4wprx              2/2     Running   0          45s   192.168.0.64   kubwor1-3   <none>           <none>
trident-node-linux-95n78              2/2     Running   0          45s   192.168.0.62   kubwor1-1   <none>           <none>
trident-operator-74978c48ff-t6qd2     1/1     Running   0          17m   10.42.0.1      kubwor1-3   <none>           <none>


In [26]:
kubectl -n trident get pods -o wide


NAME                                  READY   STATUS    RESTARTS   AGE   IP             NODE        NOMINATED NODE   READINESS GATES
trident-controller-66b97d5f8d-mjnwg   6/6     Running   0          59s   10.36.0.1      kubwor1-1   <none>           <none>
trident-node-linux-4wprx              2/2     Running   0          58s   192.168.0.64   kubwor1-3   <none>           <none>
trident-node-linux-95n78              2/2     Running   0          58s   192.168.0.62   kubwor1-1   <none>           <none>
trident-operator-74978c48ff-t6qd2     1/1     Running   0          17m   10.42.0.1      kubwor1-3   <none>           <none>


In [ ]:
kubectl -n trident get pods -o wide


---

Add the label to Worker 2:


In [27]:
kubectl label node kubwor1-2 storage=trident


node/kubwor1-2 labeled


---

Verify which nodes trident is running on:


In [28]:
kubectl -n trident get pods -o wide


NAME                                  READY   STATUS    RESTARTS   AGE   IP             NODE        NOMINATED NODE   READINESS GATES
trident-controller-66b97d5f8d-mjnwg   6/6     Running   0          95s   10.36.0.1      kubwor1-1   <none>           <none>
trident-node-linux-4wprx              2/2     Running   0          94s   192.168.0.64   kubwor1-3   <none>           <none>
trident-node-linux-95n78              2/2     Running   0          94s   192.168.0.62   kubwor1-1   <none>           <none>
trident-node-linux-dkgkw              1/2     Running   0          13s   192.168.0.63   kubwor1-2   <none>           <none>
trident-operator-74978c48ff-t6qd2     1/1     Running   0          18m   10.42.0.1      kubwor1-3   <none>           <none>


---


CHALLENGE STEP: 

You can update an existing instance of the trident CR by using a patch command. 

For example, if you want to turn off debug logs, use the following command:



In [29]:
kubectl -n trident patch torc trident --type=json -p '[{"op": "replace", "path": "/spec/debug", "value": "false"}]'


tridentorchestrator.trident.netapp.io/trident patched


---

CHALLENGE STEP: 

Verify that the debug logs are off.


---

In [30]:
kubectl -n trident describe torc trident 


Name:         trident
Namespace:    
Labels:       <none>
Annotations:  <none>
API Version:  trident.netapp.io/v1
Kind:         TridentOrchestrator
Metadata:
  Creation Timestamp:  2025-07-08T19:04:47Z
  Generation:          2
  Resource Version:    228445
  UID:                 d18633ed-9963-4736-b9e3-664b10ff1732
Spec:
  Cloud Provider:     
  Debug:              false
  Image Pull Policy:  IfNotPresent
  Namespace:          trident
  Node Plugin Node Selector:
    Storage:  trident
  Windows:    false
Status:
  Acp Version:  v24.10.0
  Current Installation Params:
    IPv6:                       false
    Acp Image:                  
    Autosupport Hostname:       
    Autosupport Image:          docker.io/netapp/trident-autosupport:24.10
    Autosupport Insecure:       false
    Autosupport Proxy:          
    Autosupport Serial Number:  
    Debug:                      true
    Disable Audit Log:          true
    Enable ACP:                 false
    Enable Force Detach:       

Turn on debug logs

Verify that the debug logs are on.


In [31]:
kubectl -n trident patch torc trident --type=json -p '[{"op": "replace", "path": "/spec/debug", "value": "true"}]'


tridentorchestrator.trident.netapp.io/trident patched


In [32]:
kubectl -n trident describe torc trident 


Name:         trident
Namespace:    
Labels:       <none>
Annotations:  <none>
API Version:  trident.netapp.io/v1
Kind:         TridentOrchestrator
Metadata:
  Creation Timestamp:  2025-07-08T19:04:47Z
  Generation:          3
  Resource Version:    228694
  UID:                 d18633ed-9963-4736-b9e3-664b10ff1732
Spec:
  Cloud Provider:     
  Debug:              true
  Image Pull Policy:  IfNotPresent
  Namespace:          trident
  Node Plugin Node Selector:
    Storage:  trident
  Windows:    false
Status:
  Acp Version:  v24.10.0
  Current Installation Params:
    IPv6:                       false
    Acp Image:                  
    Autosupport Hostname:       
    Autosupport Image:          docker.io/netapp/trident-autosupport:24.10
    Autosupport Insecure:       false
    Autosupport Proxy:          
    Autosupport Serial Number:  
    Debug:                      true
    Disable Audit Log:          true
    Enable ACP:                 false
    Enable Force Detach:        

---
---

#### Task 3: Set up the tridentctl Tool

The tridentctl tool was installed when you unzipped the trident-installer file.


---

From the Exercise 2 folder, execute [exercise2Task3.sh](./exercise2Task3.sh) file from a terminal:


In [35]:
cd ~
target_dir=$(find . -type d -name 'STRSW-ILT-UATWK-1*' 2>/dev/null | head -n 1)
if [ -n "$target_dir" ]; then
    cd "$target_dir"
else
    echo "Directory not found"
fi
cd .'/Exercise 2'

./exercise2Task3.sh


############################################
###          tridentctl install          ###
############################################
[sudo] password for user: 


---

Review the tridentctl subcommands:

tridentctl

Sample output:

```terminal
A CLI tool for managing the NetApp Trident external storage provisioner for Kubernetes
Usage:
tridentctl [command]
Available Commands:
completion Generate the autocompletion script for the specified shell
create Add a resource to Trident
delete Remove one or more resources from Trident
get Get one or more resources from Trident
help Help about any command
images Print a table of the container images Trident needs
import Import an existing resource to Trident
install Install Trident
logs Print the logs from Trident
send Send a resource from Trident
uninstall Uninstall Trident
update Modify a resource in Trident
version Print the version of Trident
Flags:
-d, --debug Set the log level to debug
-h, --help help for tridentctl
-k, --kubeconfig string Kubernetes config path
--log-level string Log level (trace, debug, warn, info, error, fatal (default "info")
-n, --namespace string Namespace of Trident deployment
-o, --output string Output format. One of json|yaml|name|wide|ps (default)
-s, --server string Address/port of Trident REST interface (127.0.0.1 or [::1] only)
Use "tridentctl [command] --help" for more information about a command.


In [36]:
tridentctl


A CLI tool for managing the NetApp Trident external storage provisioner for Kubernetes

Usage:
  tridentctl [command]

Available Commands:
  completion  Generate the autocompletion script for the specified shell
  create      Add a resource to Trident
  delete      Remove one or more resources from Trident
  get         Get one or more resources from Trident
  help        Help about any command
  images      Print a table of the container images Trident needs
  import      Import an existing resource to Trident
  install     Install Trident
  logs        Print the logs from Trident
  send        Send a resource from Trident
  uninstall   Uninstall Trident
  update      Modify a resource in Trident
  version     Print the version of Trident

Flags:
  -d, --debug               Set the log level to debug
  -h, --help                help for tridentctl
  -k, --kubeconfig string   Kubernetes config path
      --log-level string    Log level (trace, debug, warn, info, error, fatal (default "

---

Verify which version of Trident is installed:
tridentctl -n trident version

```terminal
Sample output:
+----------------+----------------+
| SERVER VERSION | CLIENT VERSION |
+----------------+----------------+
| 24.10.0 | 24.10.0 |
+----------------+----------------+
```


In [37]:
tridentctl -n trident version


+----------------+----------------+
| SERVER VERSION | CLIENT VERSION |
+----------------+----------------+
| 24.10.0        | 24.10.0        |
+----------------+----------------+


---

See the images that are required for Trident to function, per the Kubernetes version:

`tridentctl -n trident images`

```terminal
Sample output:
…
+--------------------+---------------------------------------------------------------+
| v1.31.0 | netapp/trident:24.10.0 |
| | docker.io/netapp/trident-autosupport:24.10 |
| | registry.k8s.io/sig-storage/csi-provisioner:v5.1.0 |
| | registry.k8s.io/sig-storage/csi-attacher:v4.7.0 |
| | registry.k8s.io/sig-storage/csi-resizer:v1.12.0 |
| | registry.k8s.io/sig-storage/csi-snapshotter:v8.1.0 |
| | registry.k8s.io/sig-storage/csi-node-driver-registrar:v2.12.0 |
| | netapp/trident-operator:24.10.0 (optional) |
+--------------------+---------------------------------------------------------------+


In [38]:

tridentctl -n trident images

+--------------------+---------------------------------------------------------------+
| KUBERNETES VERSION |                        CONTAINER IMAGE                        |
+--------------------+---------------------------------------------------------------+
| v1.25.0            | netapp/trident:24.10.0                                        |
|                    | docker.io/netapp/trident-autosupport:24.10                    |
|                    | registry.k8s.io/sig-storage/csi-provisioner:v5.1.0            |
|                    | registry.k8s.io/sig-storage/csi-attacher:v4.7.0               |
|                    | registry.k8s.io/sig-storage/csi-resizer:v1.12.0               |
|                    | registry.k8s.io/sig-storage/csi-snapshotter:v8.1.0            |
|                    | registry.k8s.io/sig-storage/csi-node-driver-registrar:v2.12.0 |
|                    | netapp/trident-operator:24.10.0 (optional)                    |
+--------------------+---------------------

---
---

#### Task 4: Prepare worker nodes

In this task, you verify that the worker nodes can use the volumes that Trident provides.


---

Open a Secure Shell (SSH) session to Worker 1:
ssh root@192.168.0.62

You can also use the code cells to execute the code

---

Verify that nfs-common is installed:


In [39]:
ssh root@kubwor1-1 apt list --installed | grep nfs-common




nfs-common/jammy-updates,now 1:2.6.1-1ubuntu1.2 amd64 [installed]


---

Verify that open-iscsi, lsscsi, and scsitools are installed:


In [41]:
ssh root@kubwor1-1  apt list --installed | grep scsi




libopeniscsiusr/jammy,now 2.1.5-1ubuntu1 amd64 [installed,upgradable to: 2.1.5-1ubuntu1.1]
lsscsi/jammy,now 0.31-1build2 amd64 [installed]
open-iscsi/jammy,now 2.1.5-1ubuntu1 amd64 [installed,upgradable to: 2.1.5-1ubuntu1.1]
scsitools/jammy,now 0.12-3ubuntu1 amd64 [installed]



---

Verify that sg3-utils is installed:


In [42]:
ssh root@kubwor1-1  apt list --installed | grep sg3




sg3-utils-udev/jammy,jammy,now 1.46-1build1 all [installed,upgradable to: 1.46-1ubuntu0.22.04.1]
sg3-utils/jammy,now 1.46-1build1 amd64 [installed,upgradable to: 1.46-1ubuntu0.22.04.1]


---

Verify that multipath-tools is installed:


In [43]:
ssh root@kubwor1-1  apt list --installed | grep multipath




multipath-tools/jammy-security,now 0.8.8-1ubuntu1.22.04.1 amd64 [installed,upgradable to: 0.8.8-1ubuntu1.22.04.4]


---

Verify that /etc/multipath.conf has the following values :
defaults {
user_friendly_names yes
find_multipaths no
}


In [44]:
ssh root@kubwor1-1 cat /etc/multipath.conf

defaults {
    user_friendly_names yes
}


---

Enable multipathing:


In [45]:
ssh root@kubwor1-1  systemctl enable --now iscsid multipathd


Synchronizing state of iscsid.service with SysV service script with /lib/systemd/systemd-sysv-install.
Executing: /lib/systemd/systemd-sysv-install enable iscsid
Created symlink /etc/systemd/system/sysinit.target.wants/iscsid.service → /lib/systemd/system/iscsid.service.


NOTE: If you see an error, please ignore it.


In [46]:
ssh root@kubwor1-1  "service iscsid restart && service multipathd restart"


---

Verify that multipath-tools and iscsid and are enabled and running:


In [47]:
ssh root@kubwor1-1  systemctl status multipathd iscsid


● multipathd.service - Device-Mapper Multipath Device Controller
     Loaded: loaded (/lib/systemd/system/multipathd.service; enabled; vendor preset: enabled)
     Active: active (running) since Tue 2025-07-08 16:19:18 EDT; 11s ago
TriggeredBy: ● multipathd.socket
    Process: 44361 ExecStartPre=/sbin/modprobe -a scsi_dh_alua scsi_dh_emc scsi_dh_rdac dm-multipath (code=exited, status=0/SUCCESS)
   Main PID: 44362 (multipathd)
     Status: "up"
      Tasks: 7
     Memory: 18.5M
        CPU: 50ms
     CGroup: /system.slice/multipathd.service
             └─44362 /sbin/multipathd -d -s

Jul 08 16:19:18 Kubwor1-1 systemd[1]: Starting Device-Mapper Multipath Device Controller...
Jul 08 16:19:18 Kubwor1-1 multipathd[44362]: --------start up--------
Jul 08 16:19:18 Kubwor1-1 multipathd[44362]: read /etc/multipath.conf
Jul 08 16:19:18 Kubwor1-1 multipathd[44362]: path checkers start up
Jul 08 16:19:18 Kubwor1-1 multipathd[44362]: sda: failed to get udev uid: No data available
Jul 08 16:19:18 K

---

Verify your initiator node name:


In [48]:
ssh root@kubwor1-1 cat /etc/iscsi/initiatorname.iscsi


## DO NOT EDIT OR REMOVE THIS FILE!
## If you remove this file, the iSCSI daemon will not start.
## If you change the InitiatorName, existing access control lists
## may reject this initiator.  The InitiatorName must be unique
## for each iSCSI initiator.  Do NOT duplicate iSCSI InitiatorNames.
InitiatorName=iqn.1993-08.org.debian:01:f7a5e482336


---

Verify that nvme-cli is installed:


In [49]:
ssh root@kubwor1-1  apt list --installed | grep nvme




nvme-cli/jammy-updates,now 1.16-3ubuntu0.3 amd64 [installed]


---

Scan the NVMe bus:


In [50]:
ssh root@kubwor1-1  modprobe nvme-tcp


---

Verify your NVMe Qualified Name (NQN):


In [51]:
ssh root@kubwor1-1 cat /etc/nvme/hostnqn



nqn.2014-08.org.nvmexpress:uuid:04b0b408-1ded-4528-b899-9bce6280866b


---

Implement these steps across all nodes in the source and destination clusters:

[./exercise2Task4.sh](./exercise2Task4.sh)


In [52]:

./exercise2Task4.sh

Run: kubwor1-1
Step 1
multipath.conf                                100%   64    68.2KB/s   00:00    
Step 2
Synchronizing state of iscsid.service with SysV service script with /lib/systemd/systemd-sysv-install.
Executing: /lib/systemd/systemd-sysv-install enable iscsid
Step 3
## DO NOT EDIT OR REMOVE THIS FILE!
## If you remove this file, the iSCSI daemon will not start.
## If you change the InitiatorName, existing access control lists
## may reject this initiator.  The InitiatorName must be unique
## for each iSCSI initiator.  Do NOT duplicate iSCSI InitiatorNames.
InitiatorName=iqn.1993-08.org.debian:01:f7a5e482336
Step 4
Step 5
nqn.2014-08.org.nvmexpress:uuid:04b0b408-1ded-4528-b899-9bce6280866b
Run: kubwor1-2
Step 1
multipath.conf                                100%   64    50.1KB/s   00:00    
Step 2
Synchronizing state of iscsid.service with SysV service script with /lib/systemd/systemd-sysv-install.
Executing: /lib/systemd/systemd-sysv-install enable iscsid
Created symlink /etc


End of exercise